# Data Cleaning and Preparation

Many researchers choose to do ``ad hoc`` processing of data from one form to another using a general-purpose programming language, like Python, Perl, R, or Java, or Unix text-processing tools like sed or awk. 

Here we discuss tools for missing data, duplicate data, string manipulation, and some other analytical data transformations.


## Handling Missing Data

Missing data occurs commonly in many data analysis applications. One of the goals of pandas is to make working with missing data as painless as possible. For example, all of the descriptive statistics on pandas objects exlude missing data by default. 

The way that missing data is represented in pandas objects is somewhat imperfect, but it is sufficient for most real-world use. For data with ``float64`` dtype, pandas uses the floating-point value ``NaN`` to represent missing data. 


We call this a *sentinel value*: when present, it indicates a missing (or *null*) value:

In [1]:
import numpy as np 
import pandas as pd 

float_data = pd.Series([1.2, -3.5, np.nan,0])

float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

The ``isna`` method gives us a Boolean Series with ``True`` where values are null:

In [2]:
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

When cleaning up data for analysis, it is often important to do analysis on the missing data itself to identify data collection problems biases in the data caused by missing data. 

The built-in Python ``None`` value is also treated as NA:

In [3]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])

string_data

0    aardvark
1         NaN
2        None
3     avocado
dtype: object

In [4]:
string_data.isna()

0    False
1     True
2     True
3    False
dtype: bool

In [5]:
float_data = pd.Series([1, 2, None], dtype = 'float64')

float_data

0    1.0
1    2.0
2    NaN
dtype: float64

In [6]:
float_data.isna()

0    False
1    False
2     True
dtype: bool

List of some functions related to missing data handling:

| Method | Description |
|---|---|
| `dropna` | Filter axis labels based on whether values for each label have missing data, with varying thresholds for how much missing data to tolerate. |
|``fillna``|Fill missing data with some value or using an interpolation method such as ``"ffill"`` or ``"bfill"``|
|``isna``|Return Boolen values indicating which values are missing/NA.|
|``notna``|Negation of ``isna``, returns ``True`` for non-NA values and ``False`` for NA values|


## Filtering Out Missing Data

There are a few ways to filter out missing data. While you always have the option to do it by hand using ``pandas.isna`` and Boolean indexing, ``dropna`` can be helpful. On a Series, it returns the Series with only the nonnull data and index values:

In [7]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])

data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

This is the same thing as doing:

In [8]:
data[data.notna()]

0    1.0
2    3.5
4    7.0
dtype: float64

With DataFrame objects, there are different ways to remove missing data. You may want to drop rows or columns that are all NA, or only those rows or columns containing any NAs at all. ``dropna`` by default drops any row containing a missing value:

In [9]:
data = pd.DataFrame([[1., 6.5, 3., ], [1., np.nan, np.nan], [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])

data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [10]:
data.dropna()

,0,1,2
0,1.0,6.5,3.0


Passing ``how="all"`` will drop only rows that are all NA:

In [11]:
data.dropna(how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


To drop columns in the same way, pass  ``axis = "columns"``:

In [12]:
data[4] = np.nan

data

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [13]:
data.dropna(axis="columns", how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


Suppose you want to keep only rows containing at most a certain number of missing observations. You can indicate this with the ``thresh`` argument:

In [14]:
df = pd.DataFrame(np.random.standard_normal((7, 3)))

df.iloc[:4, 1] = np.nan

df.iloc[:2, 2] = np.nan

df

,0,1,2
0,-1.645145,NaN,NaN
1,-1.440511,NaN,NaN
2,0.322132,NaN,1.917164
3,0.668318,NaN,-0.199741
4,1.131597,-1.722648,-0.519604
5,1.457740,0.089943,0.812668
6,2.226298,-1.231731,-1.846933


In [15]:
df.dropna()

,0,1,2
4,1.131597,-1.722648,-0.519604
5,1.457740,0.089943,0.812668
6,2.226298,-1.231731,-1.846933


In [16]:
df.dropna(thresh=2)

,0,1,2
2,0.322132,NaN,1.917164
3,0.668318,NaN,-0.199741
4,1.131597,-1.722648,-0.519604
5,1.457740,0.089943,0.812668
6,2.226298,-1.231731,-1.846933


## Filling in Missing Data 

Rather than filtering out missing data (and potentially discarding other data along with it), you may want to fill in the "holes" in any number of ways. For most purposes, the ``fillna`` method is the workhorse function to use. Calling ``fillna`` with a constant replaces missing values with that value:

In [17]:
df.fillna(0)

,0,1,2
0,-1.645145,0.000000,0.000000
1,-1.440511,0.000000,0.000000
2,0.322132,0.000000,1.917164
3,0.668318,0.000000,-0.199741
4,1.131597,-1.722648,-0.519604
5,1.457740,0.089943,0.812668
6,2.226298,-1.231731,-1.846933


Calling ``fillna`` with a dictionary, you can use a different full value for each column:

In [18]:
df.fillna({1: 0.5, 2:0})

,0,1,2
0,-1.645145,0.500000,0.000000
1,-1.440511,0.500000,0.000000
2,0.322132,0.500000,1.917164
3,0.668318,0.500000,-0.199741
4,1.131597,-1.722648,-0.519604
5,1.457740,0.089943,0.812668
6,2.226298,-1.231731,-1.846933


The same interpolation methods available for reindexing can be used with ``fillna``:


In [19]:
df = pd.DataFrame(np.random.standard_normal((6, 3)))

df.iloc[2:, 1] = np.nan

df.iloc[4:, 2] = np.nan

df

,0,1,2
0,-0.286197,0.565766,0.419209
1,-0.225230,1.792677,-0.750141
2,0.362959,NaN,0.801143
3,1.608039,NaN,-0.469291
4,0.441243,NaN,NaN
5,0.990286,NaN,NaN


In [20]:
df.ffill()

,0,1,2
0,-0.286197,0.565766,0.419209
1,-0.225230,1.792677,-0.750141
2,0.362959,1.792677,0.801143
3,1.608039,1.792677,-0.469291
4,0.441243,1.792677,-0.469291
5,0.990286,1.792677,-0.469291


In [21]:
df.ffill(limit=2)

,0,1,2
0,-0.286197,0.565766,0.419209
1,-0.225230,1.792677,-0.750141
2,0.362959,1.792677,0.801143
3,1.608039,1.792677,-0.469291
4,0.441243,NaN,-0.469291
5,0.990286,NaN,-0.469291


With ``fillna`` you can do lots of other things such as simple data imputation using the median and mean statistics:

In [22]:
data = pd.Series([1., np.nan, 3.5, np.nan, 7])

data.fillna(data.mean())

0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

# Data Transformation

Filtering, cleaning, and other transformations are another class of important operations. 

## Removing Duplicates 

Duplicate rows may be found in a DataFrame for any number of reasons. Here is an example:

In [23]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                    "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


The DataFrame method ``duplicated`` returns a Boolean Series indicating whether each row is a duplicate (its column values are exactly equal  to those in an earlier row) or not:

In [24]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

Relatedly, ``drop_duplcates`` returns a DataFrame with rows where the ``duplicated`` array is ``False`` filtered out:

In [25]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


Both methods by default consider all of the columns; alternatively, you can specify any subset of them to detect duplicates. Suppose we had an additional column of values and wanted to filter duplicates based on the "k1" column:

In [26]:
data["v1"] = range(7)

data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [27]:
data.drop_duplicates(subset=["k1"])

,k1,k2,v1
0,one,1,0
1,two,1,1


``duplicated`` and ``drop_duplicates`` by default keep the first observed value combination. Passing ``keep="last"`` will return the last one:

In [28]:
data.drop_duplicates(["k1", "k2"], keep="last")

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


## Transforming Data Using a Function or Mapping

For many datasets, you may wish to perform some transformation based on the values in the array, Series, or column in a DataFrame. Consider the following hypothetical data collected about various kinds of meat:

In [29]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                            "pastrami", "corned beef", "bacon",
                            "pastrami", "honey ham", "nova lox"],
                    "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


Suppose you wanted to add a column indicating the type of animal that each food came from. Let's write down a mapping of each distinct meat type to the kind of animal:

In [30]:
meat_to_animal = {
"bacon": "pig",
"pulled pork": "pig",
"pastrami": "cow",
"corned beef": "cow",
"honey ham": "pig",
"nova lox": "salmon"
}

The ``map`` method on a Series accepts a function or dictionary-like object containing a mapping to do the transformation of values:

In [31]:
data["animal"] = data["food"].map(meat_to_animal)

data 

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


We could also have passed a function that does all the work:

In [32]:
def get_animal(x):
    return meat_to_animal[x]

data["food"].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: object

Using ``map`` is a convinient way to perform element-wise transformations and other data cleaning-relation operations.

## Replacing Values

Filling in missing data with the ``fillna`` method is a special case of more general value replacement. 

``map`` can be used to modify a subset of values in an object, but ``replace`` provides a simpler and more flexible way to do so. Let's consider this series:

In [33]:
data = pd.Series([1., -999., -1000., 3.])

data 

0       1.0
1    -999.0
2   -1000.0
3       3.0
dtype: float64

The ``-999`` values might be sentinel values for missing data. To replace these with NA values that pandas understands, we can use ``replace``, producing a new Series:

In [34]:
data.replace(-999, np.nan)



0       1.0
1       NaN
2   -1000.0
3       3.0
dtype: float64

If you want to replace multiple values at once, you instead pass a list and then the substitue value:

In [35]:
data.replace([-999, -1000], np.nan)

0    1.0
1    NaN
2    NaN
3    3.0
dtype: float64

To use a different replacement for each value, pass a list of substites:

In [36]:
data.replace([-999, -1000], [np.nan, 0])

0    1.0
1    NaN
2    0.0
3    3.0
dtype: float64

The argument passed can be a dictionary:

In [37]:
data.replace({-999: np.nan, -1000:0})

0    1.0
1    NaN
2    0.0
3    3.0
dtype: float64

## Renaming Axis Indexes

Like values in a Series, axis labels can be similarly transformed by a function or mapping of some form to produce new, differently labeled objects. You can alse modify the axes in place without creating a new data structure. Here's a simple example:

In [38]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)), 
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])

data 

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


Like a Series, the axis indexes have a ``map`` method:

In [39]:
def transform(x):
    return x[:4].upper()

data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

You can assign to the ``index`` attribute, modifying the DataFrame in place:

In [40]:
data.index = data.index.map(transform)

data 

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


If you want to create a transformed version of a dataset without modifying the original, a useful method is ``rename``:

In [41]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


Notably, ``rename`` can be used in conjunction with a dictionary-like object, providing new values for a subset of the axis labels:

In [42]:
data.rename(index={"OHIO": "INDIANA"}, columns={"three": "peekaboo"})



,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


``rename`` saves you from the chore of copying the DataFrame manually and assigning new values to its ``index`` and ``columns`` attributes.


## Discretization and Binning 

Continous data is often discretized or otherwise separated into "bins" for analysis. Suppose you have data about a group of people in a study, and you want to group them into a discrete age buckets:

In [43]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

Let's divide these into bins of 18 to 25, 26 to 35, 36 to 60, and finally 61 and older. To do so, you have to use ``pandas.cut``:

In [44]:
bins = [18, 25, 35, 60, 100]

In [45]:
age_categories = pd.cut(ages, bins)

age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

The object pandas returns is a special Categorical object. The output you see describes the bins computed by ``pandas.cut``. Each bin is identified by a special interval value type containing the lower and upper limit of each bin:

In [46]:
age_categories.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [47]:
age_categories.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [48]:
age_categories.categories[0]

Interval(18, 25, closed='right')

Note that ``pd.values_counts(categories)`` are the bin counts for the result of ``pandas.cut`` 

In [49]:
pd.cut(ages, bins, right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

You can override the default interval-based bin labeling by passing a list or array to the ``labels`` option:

In [50]:
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]

pd.cut(ages, bins, labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

If you pass an integer number of bins to ``pandas.cut`` instead of explicit bin edges, it will compute equal length bins based on the minimum and maximum values in the data. Consider the case of some uniformly distributed data chopped into fourths:

In [51]:
data = np.random.uniform(size = 20)

pd.cut(data, 4, precision=2)

[(0.75, 0.97], (0.072, 0.3], (0.75, 0.97], (0.072, 0.3], (0.52, 0.75], ..., (0.3, 0.52], (0.072, 0.3], (0.072, 0.3], (0.072, 0.3], (0.75, 0.97]]
Length: 20
Categories (4, interval[float64, right]): [(0.072, 0.3] < (0.3, 0.52] < (0.52, 0.75] < (0.75, 0.97]]

The ``precision=2`` option limits the decimal precision to two digits. 

A closely related function, ``pandas.qcut`` bins the data based on sample quantiles. Depending on the distribution of the data, using ``pandas.cut`` will not usually result in each bin having the same number of data points. Since ``pandas.qcut`` uses sample quantiles instead, you will obtain roughly equally sized bins:

In [52]:
data = np.random.standard_normal(1000)

quartiles = pd.qcut(data, 4, precision=2)

quartiles

[(0.68, 3.35], (-0.67, 0.03], (0.03, 0.68], (-0.67, 0.03], (-0.67, 0.03], ..., (0.03, 0.68], (-0.67, 0.03], (0.03, 0.68], (0.68, 3.35], (0.68, 3.35]]
Length: 1000
Categories (4, interval[float64, right]): [(-2.94, -0.67] < (-0.67, 0.03] < (0.03, 0.68] < (0.68, 3.35]]

In [53]:
pd.Series(quartiles).value_counts()

(-2.94, -0.67]    250
(-0.67, 0.03]     250
(0.03, 0.68]      250
(0.68, 3.35]      250
Name: count, dtype: int64

# Detecting and Filtering Outliers 

Filtering or transforming outliers is largely a matter of applying array operations. Consider a DataFrame with some normally distributed data:

In [54]:
data = pd.DataFrame(np.random.standard_normal((1000,4)))

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.012514,-0.001250,-0.028565,-0.030822
std,0.978993,0.976813,0.996063,0.995477
min,-2.736431,-3.060669,-3.223707,-3.178222
25%,-0.675184,-0.667732,-0.677061,-0.723740
50%,0.007239,-0.013739,-0.011888,-0.026419
75%,0.642395,0.678227,0.641142,0.643072
max,2.984575,2.898055,2.903910,2.932545


Suppose you wanted to find values in one of the columns exceeding 3 in absolute value:

In [55]:
col = data[2]

col[col.abs() > 3]

865   -3.223707
Name: 2, dtype: float64

To select all rows having a value exceeding 3 or -3, you can use the ``any`` method on a Boolean DataFrame:

In [56]:
data[(data.abs()>3).any(axis="columns")]

,0,1,2,3
339,0.197063,-3.060669,1.681786,1.975913
503,0.947278,-3.002487,-0.437026,-2.138056
666,-0.838855,-1.031613,-0.196335,-3.178222
865,2.134105,-0.431282,-3.223707,2.247349
936,1.441300,-0.106660,-0.042791,-3.077470


The parentheses around ``data.abs() > 3`` are necessary in order to call the ``any`` method on the result of the comparison operation. 

Values can be set based on these criteria. Here is code to cap values outside the interval -3 to 3:

In [57]:
data[data.abs() > 3] = np.sign(data) * 3

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.012514,-0.001187,-0.028341,-0.030566
std,0.978993,0.976617,0.995369,0.994694
min,-2.736431,-3.000000,-3.000000,-3.000000
25%,-0.675184,-0.667732,-0.677061,-0.723740
50%,0.007239,-0.013739,-0.011888,-0.026419
75%,0.642395,0.678227,0.641142,0.643072
max,2.984575,2.898055,2.903910,2.932545


The statement ``np.sign(data)`` produces 1 to -1 values based on whether the values in ``data`` are positive or negative:

In [58]:
np.sign(data).head()

,0,1,2,3
0,1.0,1.0,-1.0,1.0
1,-1.0,1.0,1.0,1.0
2,1.0,1.0,1.0,1.0
3,-1.0,-1.0,-1.0,1.0
4,1.0,-1.0,-1.0,-1.0


## Permutation and Random Sampling

Permuting (randomly reordering) a Series or the rows in a DataFrame is possible using the ``numpy.random.permutation`` function. Calling ``permutation`` with the length of the axis you want to permute produces an array of integers indicating the new ordering:

In [59]:
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))

df

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


In [60]:
sampler = np.random.permutation(5)

sampler

array([3, 0, 2, 1, 4], dtype=int32)

That array can be used in ``iloc``-based indexing or the equivalent ``take`` function:

In [61]:
df.take(sampler)

,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
0,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
1,7,8,9,10,11,12,13
4,28,29,30,31,32,33,34


In [62]:
df.iloc[sampler]

,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
0,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
1,7,8,9,10,11,12,13
4,28,29,30,31,32,33,34


By invoking ``take`` with ``axis="columns"``, we could also select a permutation of the columns:

In [63]:
column_sampler = np.random.permutation(7)

column_sampler

array([6, 4, 5, 2, 3, 1, 0], dtype=int32)

In [64]:
df.take(column_sampler, axis="columns")

,6,4,5,2,3,1,0
0,6,4,5,2,3,1,0
1,13,11,12,9,10,8,7
2,20,18,19,16,17,15,14
3,27,25,26,23,24,22,21
4,34,32,33,30,31,29,28


To select a random subset without replacement (the same row cannot appear twice), you can use the ``sample`` method on Series and DataFrame:

In [65]:
df.sample(n=3)

,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6


To generate a sample *with* replacement (to allow repeat choices), pass ``replace=True`` to ``sample``:

In [66]:
choices = pd.Series([5, 7, -1, 6, 4])

choices.sample(n=10, replace=True)

2   -1
2   -1
4    4
3    6
0    5
1    7
2   -1
3    6
2   -1
3    6
dtype: int64

## Computing Indicator/Dummy Variables

Another type of transformation for statistical modeling or machine learning applications is converting a categorical variable into a *dummy* or *indicator* matrix. If a column in a Data Frame has ``k`` distinct values, you would derive a matrix or DataFrame with ``k`` columns, containing all 1s and 0s. pandas has a ``pandas.get_dummies`` function for doing so:

In [67]:
df = pd.DataFrame({"key": [":b", "b", "a", "c", "a", "b"], 
                "data1" : range(6)})

df

,key,data1
0,:b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [68]:
pd.get_dummies(df["key"])

,:b,a,b,c
0,True,False,False,False
1,False,False,True,False
2,False,True,False,False
3,False,False,False,True
4,False,True,False,False
5,False,False,True,False


In some cases you amy want to add a prefix to the columns in the indicator DataFrame, which can be then merged with the other data. ``pandas.get_dummies`` has a prefix argument for doing this:

In [69]:
dummies = pd.get_dummies(df["key"], prefix="key")

df_with_dummy = df[["data1"]].join(dummies)

df_with_dummy

,data1,key_:b,key_a,key_b,key_c
0,0,True,False,False,False
1,1,False,False,True,False
2,2,False,True,False,False
3,3,False,False,False,True
4,4,False,True,False,False
5,5,False,False,True,False


If a row in a DataFrame belongs to multiple categories, we have to use a different approach to create the dummy variables. Let's look at the MovieLens 1M dataset:

In [70]:
mnames = ["movie_id", "title", "genres"]

movies = pd.read_table("../../Datasets/movielens/movies.dat", sep="::", header=None, names=mnames, engine = "python")

movies[:10]

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children's
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


pandas has implemented a special Series method ``str.get_dummies`` that handles this scenario of multiple group membership encoded as a delimited string:

In [71]:
dummies = movies["genres"].str.get_dummies("|")

dummies.iloc[:10, :6]

,Action,Adventure,Animation,Children's,Comedy,Crime
0,0,0,1,1,1,0
1,0,1,0,1,0,0
2,0,0,0,0,1,0
3,0,0,0,0,1,0
4,0,0,0,0,1,0
5,1,0,0,0,0,1
6,0,0,0,0,1,0
7,0,1,0,1,0,0
8,1,0,0,0,0,0
9,1,1,0,0,0,0


Then, as before, you can combine this with ``movies`` while adding a ``Genre_`` to the column names in the ``dummies`` DataFrame with the ``add_prefix`` method:

In [72]:
movies_windic = movies.join(dummies.add_prefix("Genre_"))

movies_windic.iloc[0]

movie_id                                       1
title                           Toy Story (1995)
genres               Animation|Children's|Comedy
Genre_Action                                   0
Genre_Adventure                                0
Genre_Animation                                1
Genre_Children's                               1
Genre_Comedy                                   1
Genre_Crime                                    0
Genre_Documentary                              0
Genre_Drama                                    0
Genre_Fantasy                                  0
Genre_Film-Noir                                0
Genre_Horror                                   0
Genre_Musical                                  0
Genre_Mystery                                  0
Genre_Romance                                  0
Genre_Sci-Fi                                   0
Genre_Thriller                                 0
Genre_War                                      0
Genre_Western       

A useful recipe for statistical applications is to combine ``pandas.get_dummies`` with a discretization function like ``pandas,cut``:

In [73]:
np.random.seed(12345) # to make the example repeatable 

values = np.random.uniform(size=10)

values

array([0.92961609, 0.31637555, 0.18391881, 0.20456028, 0.56772503,
       0.5955447 , 0.96451452, 0.6531771 , 0.74890664, 0.65356987])

In [74]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

pd.get_dummies(pd.cut(values, bins))

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,True
1,False,True,False,False,False
2,True,False,False,False,False
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,True,False,False
6,False,False,False,False,True
7,False,False,False,True,False
8,False,False,False,True,False
9,False,False,False,True,False


## Extension Data Types

pandas was originally built upon the capabilities present in NumPy, an array computing library used primarily for working with numerical data. Many panda concepts, such as missing data, were implemented using what was available in NumPy while trying to maximize compatibility between libraries that used NumPy and pandas together. 


More recently, pandas has developed an *extension type* system allowing for new data types to be added even if they are not supported natively by NumPy. These new data types can be treated as first class alongside data coming from NumPy arrays.

Example where we create a Series of integers with a missing value:

In [75]:
s = pd.Series([1,2,3,None])

s 

0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64

In [76]:
s.dtype

dtype('float64')

Mainly for backward compatibility reasons, Series uses the legacy behavior of using a ``float64`` data type and ``np.nan`` for the missing value. We could create this Series instead using ``pandas.Int64Dtype``:

In [78]:
s = pd.Series([1, 2, 3, None], dtype=pd.Int64Dtype())

s

0       1
1       2
2       3
3    <NA>
dtype: Int64

In [79]:
s.isna()

0    False
1    False
2    False
3     True
dtype: bool

The output ``<NA>``indicates that a value is missing for an extension type array. This uses the special ``pandas.NA`` sentinel value:

In [80]:
s[3]

<NA>

We could have also used the shorthand "``Int64``" instead of ``pd.Int64Dtype()`` to specify the type. The capitalization is necessary, other it wll be a NumPy-based nonextension type:

In [81]:
s = pd.Series([1, 2, 3, None], dtype="Int64")

pandas also has an extension type specified for string data that does not use NumPy objects array:

In [ ]:
s = pd.Series(['one', 'two', None, 'three'], dtype= pd.StringDtype())

s

0      one
1      two
2     <NA>
3    three
dtype: string

# String Manipulation

Python has long been a popular raw data manipulation language in part due to its ease of use for strings and text processing. Most text operations are made simple with the string object's built-in methods. For more complex pattern matching and text manipulations, regular expressions amy be needed. pandas add to the mix by enabling you to apply string and regular expressions concisely on whole arrays of data, additionally handling the annoyance of missing data. 

## Python Built-In String Object Methods

In many string munging and scripting applications, built-in string methods are sufficient. As an example a comma-separated string can be broken into pieces with ``split``:

In [83]:
val = "a,b,     guido"

val.split(", ")

['a,b', '    guido']

```split`` is often combined with ``strip`` to trim whitespace (including line breaks):

In [85]:
pieces = [x.strip() for x in val.split(", ")]

pieces

['a,b', 'guido']

The substrings could be concatenated together with a two-colon delimiter using addition:

But this isn't a practical generic method. A faster and more Pythonic way is to pass a list or tuple to the ``join`` method on the string "::"

In [87]:
"::".join(pieces)

'a,b::guido'

Other methods are concerned with locating substrings. Using Python's ``in`` keyword is the best way to detect a substring, though ``index`` and ``find`` can also be used:

In [88]:
"guido" in val

True

In [89]:
val.index(",")

1

In [90]:
val.find(":")

-1

Relatedly, ``count`` returns the numbers of occurrences of a particular substring:

In [91]:
val.count(",")

2

## Regular Expressions

*Regular expressions* provide a flexible way to search or match (often more complex) string patterns in text. A single expression, commonly called a *regex*, is a string formed according to the regular expression language. Python's built-in ``re`` module is responsible for applying regular expression to strings; I'll give a number of examples of its use here:

The ``re`` module functions fall into three categories:
- pattern matching 
- substitution
- splitting

These are all related; a regex describes a pattern to locate in the text, which can be used for many purposes. Let's look at a simple example:

Suppose we wanted to split a string with a variable number of whitespace characters (tabs, spaces, and newlines)

The regex describing one or more whitespace characters is ``\s+``:

In [92]:
import re 

text = "foo         bar\t baz   \tqux"

re.split(r"\s+", text)

['foo', 'bar', 'baz', 'qux']

When you call ``re.split(r"\s+", text)``, the regular expression in first *compiled*, and then its ``split`` method is called on the passed text. You can compile the regex yourself with ``re.compile``, forming a reusable regex object:

In [96]:
regex = re.compile(r"\s+")

regex.split(text)

['foo', 'bar', 'baz', 'qux']